<a href="https://colab.research.google.com/github/Patricia-oliv/Processamento-de-Linguagem-Natural-NLP-/blob/main/C%C3%B3pia_de_Entra_21_Qwen2_5_7B_Instruct_no_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torchvision

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA: True


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Você é um assistente didático e responde em português brasileiro."
    },
    {
        "role": "user",
        "content": "Explique em poucas palavras o que é processamento de linguagem natural."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

O processamento de linguagem natural é uma área da inteligência artificial que permite que computadores compreendam, analisem e geram linguagem humana de forma eficiente.


In [ ]:
def perguntar(prompt, max_tokens=500):
    messages = [
        {
            "role": "system",
            "content": "Você é um assistente didático e responde em português brasileiro."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [ ]:
print(
    perguntar(
        "Qual é a diferença entre tokenização e lematização?"
    )
)

A tokenização e a lematização são duas etapas importantes no pré-processamento de texto para tarefas de processamento de linguagem natural (NLP). Vamos entender cada uma delas:

### Tokenização

A **tokenização** é o processo de dividir um texto em unidades menores, chamadas de tokens. Um token pode ser uma palavra, um número, um sinal de pontuação, etc. O objetivo principal da tokenização é transformar o texto original em uma sequência de palavras ou tokens que podem ser facilmente processados pelo algoritmo.

**Exemplo:**
- Texto original: "O gato azul está dormindo."
- Tokenização: ["O", "gato", "azul", "está", "dormindo", "."]

### Lematização

A **lematização** é o processo de reduzir as palavras a suas formas base (lemmas) ou raízes léxicas. Isso significa que palavras comumente usadas em diferentes formas (por exemplo, conjugadas em tempo verbal ou em diferentes casos gramaticais) são reduzidas à sua forma mais básica.

**Exemplo:**
- Palavra original: "correndo"
- Lematização: 

In [ ]:
prompt = """
Analise o comentário abaixo.

Comentário:
"O lanche estava ótimo, a batata veio fria e o entregador foi muito educado."

Identifique:

1. aspectos mencionados;
2. sentimento associado a cada aspecto;
3. sentimento geral.

Responda de forma organizada.
"""

print(perguntar(prompt))

Claro, vou analisar o comentário de forma estruturada.

1. **Aspectos mencionados:**
   - Lanche
   - Batata
   - Entregador

2. **Sentimento associado a cada aspecto:**
   - **Lanche:** Ótimo (sentimento positivo)
   - **Batata:** Fria (sentimento negativo)
   - **Entregador:** Muito educado (sentimento positivo)

3. **Sentimento geral:**
   - O sentimento geral do comentário é misto, pois há tanto aspectos positivos quanto negativos mencionados. No entanto, o sentimento final parece mais positivo, pois o lanche foi elogiado e o entregador foi considerado educado, que são aspectos mais importantes na experiência de entrega de lanches.

Em conclusão, o comentário indica uma experiência geral positiva, mas com algumas nuances negativas sobre a batata fria.


In [ ]:
import torch

print(
    f"Memória alocada: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"Memória reservada: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

Memória alocada: 8.33 GB
Memória reservada: 10.58 GB
GPU: Tesla T4
